<a href="https://colab.research.google.com/github/priyal6/AI-AGENT/blob/main/evals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install deepeval openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.4/823.4 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 3.4 MB/s eta 0:00:00


In [ ]:
!pip install deepeval

In [ ]:
from openai import OpenAI
from deepeval.tracing import observe
from deepeval.dataset import Golden, EvaluationDataset
from deepeval.metrics import TaskCompletionMetric
import sys # Import sys module
import os # Import os module to access environment variables

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])




@observe()
def trip_planner_agent(user_input: str) -> str:
    """LLM-powered travel assistant"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful travel planner."},
            {"role": "user", "content": user_input},
        ],
        temperature=0.3,
    )

    return response.choices[0].message.content




goldens = [
    Golden(
        input="Plan a 2-day trip to Paris",
        expected_output="A useful itinerary for Paris"
    ),
    Golden(
        input="Romantic weekend in Venice",
        expected_output="Romantic travel plan"
    ),
    Golden(
        input="Family vacation ideas in London",
        expected_output="Family-friendly travel suggestions"
    ),
    Golden(
        input="Budget trip to Japan",
        expected_output="Low-cost travel ideas"
    ),
]

dataset = EvaluationDataset(goldens=goldens)




task_metric = TaskCompletionMetric(
    threshold=0.7,
    model="gpt-4o"   # Judge model
)




sys.__stdout__.write("\n🚀 Running DeepEval...\n")

for golden in dataset.evals_iterator(metrics=[task_metric]):

    output = trip_planner_agent(golden.input)

    # Use sys.__stdout__.write to bypass rich's proxy and avoid recursion
    sys.__stdout__.write(f"INPUT: {golden.input}\n")
    sys.__stdout__.write(f"OUTPUT: {output}\n")
    sys.__stdout__.write("-" * 50 + "\n")
    sys.__stdout__.flush() # Ensure output is written immediately


Output()



Metrics Summary

  - ✅ Task Completion (score: 1.0, threshold: 0.7, strict: False, evaluation model: gpt-4o, reason: The system provided a comprehensive list of family vacation ideas in London, covering a wide range of activities and attractions suitable for families. It also included practical tips for visiting London with family, ensuring a well-rounded and informative response that fully meets the task requirements., error: None)
  - ✅ Task Completion (score: 1.0, threshold: 0.7, strict: False, evaluation model: gpt-4o, reason: The system provided a comprehensive list of family vacation ideas in London, covering a wide range of activities and attractions suitable for families. It also included practical tips for visiting London with family, ensuring a well-rounded and informative response that fully meets the task requirements., error: None)

For test case:

  - input: Family vacation ideas in London
  - actual output: London is a fantastic destination for a family vacation, offer

⚠ WARNING: No hyperparameters logged.
» ]8;id=807762;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 79.97s | token cost: 0.21770499999999998 USD)
» Test Results (4 total tests):
   » Pass Rate: 100.0% | Passed: 4 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [ ]:
from deepeval.test_case import LLMTestCase, ToolCall
from deepeval.metrics import ToolCorrectnessMetric

In [ ]:
#tool log

tool_log = []

def log_tool(tool_name, tool_function, *args, **kwargs):
  tool_log.append(ToolCall(name= tool_name))

  return tool_function(*args, **kwargs)

In [ ]:
def web_search(query):
  return "Refund policy: 30-day return"

def inventory_database(product):
  return "Stock information"

In [ ]:
#agent wrapper

def support_agent(question):
  tool_log.clear()

  policy = log_tool("WebSearch", web_search, question)

  stock = log_tool("InventoryDatabase", inventory_database, "shoes")

  response = "You can return the shoes within 30 days"

  return response, tool_log

In [ ]:
response, tools_used = support_agent(
    "What if these shoes don't fit?"
)

print("Response: ", response)
print("Tool used: ", tools_used)

Response:  You can return the shoes within 30 days
Tool used:  [ToolCall(
    name="WebSearch"
), ToolCall(
    name="InventoryDatabase"
)]


In [ ]:
#toolcorrectness
test_case = LLMTestCase(
    input="What if these shoes don't fit?",
    actual_output= response,
    tools_called= tools_used,
    expected_tools=[ToolCall(name="WebSearch")]
)

metric = ToolCorrectnessMetric()
metric.measure(test_case)

print("Score:", metric.score)
print("Reason:", metric.reason)

Output()

Score: 1.0
Reason: [
	 Tool Calling Reason: All expected tools ['WebSearch'] were called (order not considered).
	 Tool Selection Reason: No available tools were provided to assess tool selection criteria
]



In [ ]:
!pip install langchain langchain-openai deepeval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 3.1 MB/s eta 0:00:00


In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import ToolMessage

from deepeval.test_case import ToolCall, LLMTestCase
from deepeval.metrics import ToolCorrectnessMetric


def get_weather(city: str) -> str:
  """Get weather for a given city"""
  return f"It's always sunny in {city}"

class ToolLogger(BaseCallbackHandler):
  def __init__(self):
    self.tools = []

  def on_tool_start(self, serialized, input_str, **kwargs):
    self.tools.append(ToolCall(name=serialized["name"]))

logger = ToolLogger()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a helpful assistant"
)

logger.tools.clear()

result = agent.invoke(
    {'messages': [{'role': 'user', 'content':'Weather in San Francisco?'}]},
    config={'callbacks': [logger]}
)

output_text = result['messages'][-1].content

# Extract tools used directly from the result messages
tools_used_from_result = []
for message in result['messages']:
    if isinstance(message, ToolMessage):
        tools_used_from_result.append(ToolCall(name=message.name))

print('Agent response: ', output_text)
print('Tool used: ', tools_used_from_result)


test_case = LLMTestCase(
    input="Weather in San Francisco?",
    actual_output=output_text,
    tools_called=tools_used_from_result,
    expected_tools=[ToolCall(name="get_weather")]
)

metric = ToolCorrectnessMetric()
metric.measure(test_case)

print("Score:", metric.score)
print("Reason:", metric.reason)

Output()

Agent response:  The weather in San Francisco is currently sunny.
Tool used:  [ToolCall(
    name="get_weather"
)]


Score: 1.0
Reason: [
	 Tool Calling Reason: All expected tools ['get_weather'] were called (order not considered).
	 Tool Selection Reason: No available tools were provided to assess tool selection criteria
]



In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import ToolMessage, AIMessage, HumanMessage

from deepeval import evaluate
from deepeval.test_case import ToolCall, LLMTestCase
from deepeval.metrics import (
    ToolCorrectnessMetric,
    ArgumentCorrectnessMetric,
    TaskCompletionMetric
)

import json


def get_weather(city: str, unit: str = "celsius") -> str:
    """Get weather for a city"""

    if unit.lower() == "fahrenheit":
        return f"Weather in {city} is 68°F (assuming 20C was intended as a pleasant temperature, converted to F)"
    return f"Weather in {city} is 20° {unit}"



llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You MUST use get_weather for weather questions.",
)



query = "What is the weather in Paris in Fahrenheit?"

result = agent.invoke(
    {"messages": [HumanMessage(content=query)]} # Use HumanMessage
)

output_text = result["messages"][-1].content

print("Agent response:", output_text)



tools_used = []


for msg in result["messages"]:
    if isinstance(msg, AIMessage) and msg.additional_kwargs.get("tool_calls"):
        for tool_call_data in msg.additional_kwargs["tool_calls"]:

            arguments = json.loads(tool_call_data["function"]["arguments"])
            tools_used.append(
                ToolCall(
                    name=tool_call_data["function"]["name"],
                    arguments=arguments
                )
            )

print("Tools used:", tools_used)



expected_tools = [
    ToolCall(
        name="get_weather",
        arguments={"city": "Paris", "unit": "fahrenheit"}
    )
]



test_case = LLMTestCase(
    input=query,
    actual_output=output_text,
    tools_called=tools_used,
    expected_tools=expected_tools,
)


tool_metric = ToolCorrectnessMetric()
arg_metric = ArgumentCorrectnessMetric(
    model="gpt-4o-mini",
    include_reason=True
)
task_metric = TaskCompletionMetric(
    model="gpt-4o-mini"
)



evaluate(
    test_cases=[test_case],
    metrics=[tool_metric, arg_metric, task_metric]
)


Agent response: The weather in Paris is 68°F.
Tools used: []


✨ You're running DeepEval's latest Tool Correctness Metric! (using None, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Argument Correctness Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Task Completion Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases




Metrics Summary

  - ❌ Tool Correctness (score: 0.0, threshold: 0.5, strict: False, evaluation model: None, reason: [
	 Tool Calling Reason: Incomplete tool usage: missing tools [ToolCall(
    name="get_weather"
)]; expected ['get_weather'], called []. See more details above.
	 Tool Selection Reason: No available tools were provided to assess tool selection criteria
]
, error: None)
  - ✅ Argument Correctness (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: No tool calls provided, error: None)
  - ✅ Task Completion (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The system successfully retrieved the current weather in Paris and provided it in Fahrenheit, fully aligning with the desired task., error: None)

For test case:

  - input: What is the weather in Paris in Fahrenheit?
  - actual output: The weather in Paris is 68°F.
  - expected output: None
  - context: None
  - retrieval context: None


Overall Metric Pass

⚠ WARNING: No hyperparameters logged.
» ]8;id=175200;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 2.38s | token cost: 0.00019605000000000002 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=False, metrics_data=[MetricData(name='Tool Correctness', threshold=0.5, success=False, score=0.0, reason='[\n\t Tool Calling Reason: Incomplete tool usage: missing tools [ToolCall(\n    name="get_weather"\n)]; expected [\'get_weather\'], called []. See more details above.\n\t Tool Selection Reason: No available tools were provided to assess tool selection criteria\n]\n', strict_mode=False, evaluation_model=None, error=None, evaluation_cost=0.0, verbose_logs='Expected Tools:\n[\n    ToolCall(\n        name="get_weather"\n    )\n] \n \nTools Called:\n[\n\n] \n \nAvailable Tools: [] \n \nTool Selection Score: 1.0 \n \nTool Selection Reason: No available tools were provided to assess tool selection criteria'), MetricData(name='Argument Correctness', threshold=0.5, success=True, score=1.0, reason='No tool calls provided', strict_mode=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=0.0, verbose_logs='Ver